# Unit 7 Lecture 2: Multi-Body System Simulation (Student)

## Objectives
- Formulate equations of motion for multi-body systems
- Implement constraint enforcement in simulations
- Handle chaotic dynamics (double pendulum)
- Simulate practical mechanisms (four-bar linkage)
- Validate simulation results

## Why This Matters
Real engineering systems involve multiple interconnected bodies:
- Robot arms and manipulators
- Vehicle suspension systems
- Manufacturing mechanisms
- Biomechanical systems
- Spacecraft with flexible appendages

## Overview
**Part A**: Formulation - converting multi-body dynamics to ODEs  
**Part B**: Double Pendulum - classical chaotic system  
**Part C**: Four-Bar Linkage - practical mechanism simulation  
**Duration**: ~90 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Part A: Multi-Body System Formulation

### The Challenge
Multi-body systems have:
- **Multiple degrees of freedom**: Each body contributes DOFs
- **Kinematic constraints**: Joint connections, contact conditions
- **Coupled equations**: Bodies interact through forces/torques

### Approach 1: Lagrangian Mechanics
For $n$ generalized coordinates $q_i$:
$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_i}\right) - \frac{\partial L}{\partial q_i} = Q_i$$

where $L = T - V$ (Lagrangian), $Q_i$ are generalized forces.

**Advantages**:
- Automatic handling of constraints
- Systematic formulation
- Energy-based approach

### Approach 2: Newton-Euler + Constraints
1. Write Newton's laws for each body
2. Add constraint forces
3. Solve resulting DAE (Differential-Algebraic Equation)

### Converting to First-Order ODEs
Multi-body system:
$$M(q)\ddot{q} = F(q, \dot{q}, t)$$

Convert to first-order:
$$\begin{bmatrix} \dot{q} \\ \ddot{q} \end{bmatrix} = \begin{bmatrix} \dot{q} \\ M^{-1}F \end{bmatrix}$$

State vector: $\mathbf{y} = [q_1, q_2, ..., \dot{q}_1, \dot{q}_2, ...]^T$

In [ ]:
# Example 1: Double Pendulum - Chaotic Dynamics
print("="*70)
print("Example 1: Double Pendulum Simulation")
print("="*70)

# System parameters
m1 = 1.0    # kg (first mass)
m2 = 1.0    # kg (second mass)
L1 = 1.0    # m (first length)
L2 = 1.0    # m (second length)
g = 9.81    # m/s²

print(f"\nDouble pendulum parameters:")
print(f"m₁ = {m1} kg, L₁ = {L1} m")
print(f"m₂ = {m2} kg, L₂ = {L2} m")

# Lagrangian formulation leads to coupled ODEs
def double_pendulum_ode(t, y):
    """
    Double pendulum equations of motion
    State: y = [θ₁, θ₂, ω₁, ω₂]
    """
    theta1, theta2, omega1, omega2 = y
    
    # Useful terms
    delta = theta2 - theta1
    den1 = (m1 + m2)*L1 - m2*L1*np.cos(delta)**2
    den2 = (L2/L1)*den1
    
    # Accelerations from Lagrangian equations
    dtheta1_dt = omega1
    dtheta2_dt = omega2
    
    domega1_dt = (m2*L1*omega1**2*np.sin(delta)*np.cos(delta)
                  + m2*g*np.sin(theta2)*np.cos(delta)
                  + m2*L2*omega2**2*np.sin(delta)
                  - (m1+m2)*g*np.sin(theta1)) / den1
    
    domega2_dt = (-m2*L2*omega2**2*np.sin(delta)*np.cos(delta)
                  + (m1+m2)*g*np.sin(theta1)*np.cos(delta)
                  - (m1+m2)*L1*omega1**2*np.sin(delta)
                  - (m1+m2)*g*np.sin(theta2)) / den2
    
    return [dtheta1_dt, dtheta2_dt, domega1_dt, domega2_dt]

# Initial conditions - slightly different to show chaos
y0_1 = [np.pi/2, np.pi/2, 0, 0]      # Initial: both at 90°
y0_2 = [np.pi/2, np.pi/2 + 0.01, 0, 0]  # Tiny perturbation

t_span = (0, 20)
t_eval = np.linspace(0, 20, 2000)

print(f"\nSimulating two cases:")
print(f"Case 1: θ₁ = 90°, θ₂ = 90°")
print(f"Case 2: θ₁ = 90°, θ₂ = 90.573° (0.01 rad difference)")
print(f"\nDemonstrating sensitivity to initial conditions (chaos)...")

# Solve both cases
sol1 = solve_ivp(double_pendulum_ode, t_span, y0_1, method='DOP853',
                 t_eval=t_eval, rtol=1e-10, atol=1e-13)
sol2 = solve_ivp(double_pendulum_ode, t_span, y0_2, method='DOP853',
                 t_eval=t_eval, rtol=1e-10, atol=1e-13)

print(f"Solution 1: {sol1.nfev} function evaluations")
print(f"Solution 2: {sol2.nfev} function evaluations")

# Calculate Cartesian positions
def pendulum_positions(theta1, theta2):
    """Convert angles to Cartesian coordinates"""
    x1 = L1 * np.sin(theta1)
    y1 = -L1 * np.cos(theta1)
    x2 = x1 + L2 * np.sin(theta2)
    y2 = y1 - L2 * np.cos(theta2)
    return x1, y1, x2, y2

x1_1, y1_1, x2_1, y2_1 = pendulum_positions(sol1.y[0], sol1.y[1])
x1_2, y1_2, x2_2, y2_2 = pendulum_positions(sol2.y[0], sol2.y[1])

# Calculate energy
def pendulum_energy(y):
    """Total energy of double pendulum"""
    theta1, theta2, omega1, omega2 = y
    
    # Kinetic energy
    T1 = 0.5 * m1 * (L1*omega1)**2
    T2 = 0.5 * m2 * ((L1*omega1)**2 + (L2*omega2)**2 
                     + 2*L1*L2*omega1*omega2*np.cos(theta1-theta2))
    T = T1 + T2
    
    # Potential energy (measure from pivot)
    V1 = -m1 * g * L1 * np.cos(theta1)
    V2 = -m2 * g * (L1*np.cos(theta1) + L2*np.cos(theta2))
    V = V1 + V2
    
    return T + V

E1 = np.array([pendulum_energy(sol1.y[:, i]) for i in range(len(sol1.t))])
E2 = np.array([pendulum_energy(sol2.y[:, i]) for i in range(len(sol2.t))])
E0 = E1[0]

# Visualization
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Angle evolution
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(sol1.t, np.degrees(sol1.y[0]), 'b-', linewidth=2, label='θ₁ - Case 1', alpha=0.7)
ax1.plot(sol1.t, np.degrees(sol1.y[1]), 'r-', linewidth=2, label='θ₂ - Case 1', alpha=0.7)
ax1.plot(sol2.t, np.degrees(sol2.y[0]), 'b--', linewidth=1.5, label='θ₁ - Case 2', alpha=0.7)
ax1.plot(sol2.t, np.degrees(sol2.y[1]), 'r--', linewidth=1.5, label='θ₂ - Case 2', alpha=0.7)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Angle (degrees)')
ax1.set_title('Double Pendulum: Angle vs Time (Chaos!)')
ax1.legend(ncol=4, fontsize=9)
ax1.grid(True, alpha=0.3)

# Phase space - first pendulum
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(np.degrees(sol1.y[0]), sol1.y[2], 'b-', linewidth=1, alpha=0.6)
ax2.set_xlabel('θ₁ (degrees)')
ax2.set_ylabel('ω₁ (rad/s)')
ax2.set_title('Phase Portrait: Pendulum 1')
ax2.grid(True, alpha=0.3)

# Trajectory in space
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(x2_1, y2_1, 'b-', linewidth=0.5, alpha=0.4, label='Case 1')
ax3.plot(x2_1[0], y2_1[0], 'go', markersize=8, label='Start')
ax3.plot(x2_1[-1], y2_1[-1], 'ro', markersize=8, label='End')
ax3.set_xlabel('x (m)')
ax3.set_ylabel('y (m)')
ax3.set_title('Trajectory of 2nd Mass - Case 1')
ax3.axis('equal')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# Trajectory comparison
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(x2_1, y2_1, 'b-', linewidth=0.5, alpha=0.4, label='Case 1')
ax4.plot(x2_2, y2_2, 'r-', linewidth=0.5, alpha=0.4, label='Case 2')
ax4.set_xlabel('x (m)')
ax4.set_ylabel('y (m)')
ax4.set_title('Trajectory Comparison (Tiny IC Difference)')
ax4.axis('equal')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# Divergence
ax5 = fig.add_subplot(gs[1, 2])
divergence = np.sqrt((x2_1 - x2_2)**2 + (y2_1 - y2_2)**2)
ax5.semilogy(sol1.t, divergence, 'k-', linewidth=2)
ax5.set_xlabel('Time (s)')
ax5.set_ylabel('Position Difference (m)')
ax5.set_title('Exponential Divergence (Chaos)')
ax5.grid(True, alpha=0.3, which='both')

# Energy conservation
ax6 = fig.add_subplot(gs[2, :2])
energy_error1 = (E1 - E0) / np.abs(E0) * 100
energy_error2 = (E2 - E0) / np.abs(E0) * 100
ax6.plot(sol1.t, energy_error1, 'b-', linewidth=2, label='Case 1', alpha=0.7)
ax6.plot(sol2.t, energy_error2, 'r--', linewidth=2, label='Case 2', alpha=0.7)
ax6.set_xlabel('Time (s)')
ax6.set_ylabel('Energy Error (%)')
ax6.set_title('Energy Conservation Check')
ax6.legend()
ax6.grid(True, alpha=0.3)

# Statistics
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('off')
stats_text = f"""CHAOS ANALYSIS
━━━━━━━━━━━━━━━━━━━━
Initial difference:
  Δθ₂ = 0.573°

After 20 seconds:
  Position diff: {divergence[-1]:.3f} m
  Divergence: {divergence[-1]/0.01:.0f}× initial

Energy conservation:
  Max error: {np.max(np.abs(energy_error1)):.4f}%
  
Lyapunov time: ~3-4 s
(time to e-fold divergence)

Conclusion:
System is chaotic - tiny
changes lead to completely
different trajectories!"""
ax7.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
         verticalalignment='center')

plt.savefig('notebooks/Teacher/figs/u7_l2_double_pendulum.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*70}")
print("Key Observations:")
print(f"• Initial difference: 0.01 rad (0.573°)")
print(f"• Final divergence: {divergence[-1]:.3f} m")
print(f"• Energy conserved to: {np.max(np.abs(energy_error1)):.4f}%")
print(f"• Double pendulum is chaotic!")
print("="*70)
# TODO: Use these parameters in your solutions below


## Part C: Four-Bar Linkage Mechanism

### Practical Mechanism
Four-bar linkages are ubiquitous in engineering:
- **Applications**: Windshield wipers, car suspension, robotic grippers, door closers
- **Advantages**: Simple, robust, predictable motion
- **Challenge**: Constrained motion requires careful formulation

### Configuration
```
        Link 2 (driven)
    B ●━━━━━━━━● C
      ╱          ╲
Link 1╱            ╲Link 3
    ╱                ╲
  A●━━━━━━━━━━━━━━━━●D
        Link 4 (ground)
```

### Kinematic Loop Constraint
The four links form a closed loop:
$$\vec{L}_1 + \vec{L}_2 + \vec{L}_3 + \vec{L}_4 = 0$$

In component form:
$$\begin{align}
L_1\cos\theta_1 + L_2\cos\theta_2 + L_3\cos\theta_3 - L_4 &= 0 \\
L_1\sin\theta_1 + L_2\sin\theta_2 + L_3\sin\theta_3 &= 0
\end{align}$$

### Simulation Approach
1. **Input**: Drive link 1 at constant angular velocity
2. **Constraint**: Solve loop equation for $\theta_2, \theta_3$
3. **Dynamics**: Include inertia and solve for torques
4. **Validation**: Check constraint violation

In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

## Summary

### Multi-Body Simulation Workflow

1. **Formulation**
   - Choose coordinates (Lagrangian or Newton-Euler)
   - Identify constraints
   - Derive equations of motion
   - Convert to first-order ODEs

2. **Implementation**
   - Write ODE function
   - Choose integrator (RK45 for most cases)
   - Set tight tolerances for accuracy
   - Include constraint enforcement if needed

3. **Validation**
   - Check energy conservation
   - Verify constraint satisfaction
   - Compare with analytical solutions (if available)
   - Visualize motion for physical reasonableness

### Double Pendulum Insights

- **Chaotic system**: Exponential sensitivity to initial conditions
- **Lyapunov time**: ~3-4 seconds for e-fold divergence
- **Energy conservation**: Critical for long simulations
- **High accuracy required**: Use DOP853 or tight tolerances

### Four-Bar Linkage Insights

- **Grashof condition**: Determines if crank can rotate fully
- **Kinematic loop**: Closed-form solution possible
- **Non-uniform motion**: Output velocity varies with input
- **Practical applications**: Simple, robust mechanism design

### Simulation Best Practices

**Choose appropriate method**:
- Non-stiff systems: `RK45` or `DOP853`
- Stiff systems: `Radau` or `BDF`
- Unknown: `LSODA` (auto-switching)

**Set tolerances carefully**:
```python
rtol = 1e-8   # Relative tolerance
atol = 1e-11  # Absolute tolerance
```

**Validate results**:
- Energy conservation (conservative systems)
- Constraint satisfaction (multi-body)
- Physical reasonableness (visualization)

**Computational efficiency**:
- Use vectorized operations
- Avoid unnecessary calculations in ODE function
- Consider adaptive vs fixed step for speed/accuracy trade-off

### Next: Practical Application
In the practical, we'll combine these concepts to simulate a complete system with:
- Multiple bodies and DOFs
- External forcing
- Constraint enforcement
- Results visualization and analysis